# Introduction to AI: E-commerce Session Segmentation and Purchase Prediction
This notebook complements the lecture slides in `slides/introduction_to_ai/slides.md`.

## Story
An online shop wants to understand **session segments** (unsupervised learning) and **predict purchases** (supervised learning). We use a real e‑commerce dataset of web sessions.

**Dataset (Online Shoppers Purchasing Intention, UCI)**
Each row is a web session. The label `revenue` indicates whether the session ended with a purchase.

**Goal:** connect clustering and classification to a clear, intuitive e‑commerce use case.

## Exercises
- Exercise 1: Explore and prepare the dataset (clean columns, define features/label).
- Exercise 2: Cluster sessions with k-means and interpret segments.
- Exercise 3: Train a classifier to predict `revenue` and evaluate it.

Notes:
- Data loading is already provided.
- Work through the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"

# Load the data (students should not have to do this)
df_raw = pd.read_csv(DATA_URL)

df_raw.head()

In [ ]:
# Exercise 1: Explore and prepare the data

df = df_raw.copy()

df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

rename_map = {
    "productrelated": "product_related",
    "productrelated_duration": "product_related_duration",
    "bouncerates": "bounce_rates",
    "exitrates": "exit_rates",
    "pagevalues": "page_values",
    "specialday": "special_day",
    "operatingsystems": "operating_systems",
    "traffictype": "traffic_type",
    "visitortype": "visitor_type",
}

rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
if rename_map:
    df = df.rename(columns=rename_map)

display(df.head())
display(df.isna().sum())
print(df.shape)

# Convert booleans to integers (TRUE/FALSE -> 1/0)
for col in ["weekend", "revenue"]:
    df[col] = df[col].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)

feature_cols = [
    "administrative",
    "administrative_duration",
    "informational",
    "informational_duration",
    "product_related",
    "product_related_duration",
    "bounce_rates",
    "exit_rates",
    "page_values",
    "special_day",
    "weekend",
]

X = df[feature_cols]
y = df["revenue"]

display(X.describe())
y.value_counts()

## Part A: Unsupervised learning - k-means clustering
We ignore the labels and group sessions purely by behavior. This mimics a real **segmentation** use case.

### Tasks
1. Scale the features.
2. Fit k-means with a chosen k (start with k=4).
3. Inspect cluster sizes and interpret clusters using feature averages.

In [ ]:
# Exercise 2: K-means clustering

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 4
kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["cluster"] = clusters

display(df_clustered["cluster"].value_counts().sort_index())
display(df_clustered.groupby("cluster")[feature_cols].mean())

In [ ]:
# Visualize clusters in 2D with PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_plot = df_clustered.copy()
df_plot["pc1"] = X_pca[:, 0]
df_plot["pc2"] = X_pca[:, 1]

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_plot["pc1"],
    df_plot["pc2"],
    c=df_plot["cluster"],
    cmap="viridis",
    alpha=0.8,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters (PCA projection)")
plt.colorbar(scatter, label="cluster")
plt.show()

## Part B: Supervised learning - classification
Now we use the real label `revenue` and train a model to **predict** whether a session ends in a purchase.

### Tasks
1. Split the data into train and test sets.
2. Build a pipeline with scaling + a classifier.
3. Train, predict, and evaluate.

In [ ]:
# Exercise 3: Classification

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=300)),
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Plot the confusion matrix for your classifier

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## Discussion and extensions
- Try different values of k and justify your choice.
- Compare clusters to `revenue` and discuss mismatches.
- Swap the classifier (DecisionTree, RandomForest, SVM) and compare metrics.
- Compare performance with and without feature scaling.
- Which extra e‑commerce features would improve this model (recency, frequency, basket size)?